# 02 — Train Stage 1 Detector

Fine-tune **YOLOv8s** on the combined `detector_yolo` dataset.

| Setting | Value |
|---|---|
| Base model | YOLOv8s (COCO pre-trained) |
| Classes | pothole (0), traffic_light (1) |
| Epochs | 100, patience=20 |
| Image size | 640×640 |
| Augmented data | Yes (weather effects) |

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path('..').resolve()
print('Project root:', BASE_DIR)
print('GPU available:', end=' ')
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Verify Data is Ready

In [ ]:
DATA_YAML = BASE_DIR / 'data' / 'processed' / 'detector_yolo_aug' / 'dataset.yaml'
if not DATA_YAML.exists():
    DATA_YAML = BASE_DIR / 'data' / 'processed' / 'detector_yolo' / 'dataset.yaml'
    print('Using non-augmented dataset (run augment.py for weather augmentation)')

assert DATA_YAML.exists(), (
    f'Dataset YAML not found: {DATA_YAML}\n'
    'Run: python src/prepare_data.py && python src/augment.py'
)
print(f'Dataset: {DATA_YAML}')
print(DATA_YAML.read_text())

## 2. Train

In [ ]:
# Uses train.py so the result is identical to running from the command line
from train import train_detector

results = train_detector(use_aug=True, run_eval=False)
print('Training complete.')

## 3. Training Curves

In [ ]:
results_csv = BASE_DIR / 'runs' / 'detector' / 'results.csv'
assert results_csv.exists(), 'results.csv not found — did training complete?'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_pairs = [
    ('train/box_loss', 'val/box_loss',    'Box Loss'),
    ('train/cls_loss', 'val/cls_loss',    'Class Loss'),
    ('train/dfl_loss', 'val/dfl_loss',    'DFL Loss'),
    ('metrics/precision(B)', None,         'Precision'),
    ('metrics/recall(B)',    None,         'Recall'),
    ('metrics/mAP50(B)',     'metrics/mAP50-95(B)', 'mAP'),
]

for ax, (train_col, val_col, title) in zip(axes.flatten(), plot_pairs):
    if train_col in df.columns:
        ax.plot(df['epoch'], df[train_col], label='train', color='#3498db')
    if val_col and val_col in df.columns:
        ax.plot(df['epoch'], df[val_col], label='val', color='#e74c3c', linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Stage 1 Detector — Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Final Metrics on Validation Set

In [ ]:
model_path = BASE_DIR / 'models' / 'detector_model.pt'
assert model_path.exists(), 'detector_model.pt not found'

model   = YOLO(str(model_path))
orig_yaml = BASE_DIR / 'data' / 'processed' / 'detector_yolo' / 'dataset.yaml'
metrics = model.val(data=str(orig_yaml))

print(f'\nmAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

## 5. Per-Class Metrics

In [ ]:
class_names = ['pothole', 'traffic_light']
colors      = ['#e74c3c', '#2ecc71']

# metrics.box.ap_class_index gives class indices
if hasattr(metrics.box, 'ap_class_index'):
    ap50_per_class = metrics.box.ap50
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(class_names, ap50_per_class, color=colors)
    for i, v in enumerate(ap50_per_class):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title('AP@0.5 per Class')
    ax.set_ylabel('AP@0.5')
    plt.tight_layout()
    plt.show()
else:
    print('Per-class AP not available — check ultralytics version')

## 6. Sample Predictions on Validation Images

In [ ]:
val_dir   = BASE_DIR / 'data' / 'processed' / 'detector_yolo' / 'images' / 'val'
val_imgs  = sorted(val_dir.glob('*'))[:6]
class_names = ['pothole', 'traffic_light']
box_colors  = {0: (255, 80,  80),  1: (80, 220, 80)}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flatten(), val_imgs):
    frame  = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    preds  = model(img_path, conf=0.40, verbose=False)
    for box in preds[0].boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        c    = tuple(v/255 for v in box_colors.get(cls, (200,200,200)))
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=c, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(y1-4, 0), f'{class_names[cls]} {conf:.0%}',
                color=c, fontsize=7, fontweight='bold')
    ax.imshow(frame)
    ax.axis('off')
    ax.set_title(img_path.name[:35], fontsize=7)

plt.suptitle('Stage 1 Detector — Val Set Predictions (conf≥0.40)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Confusion Matrix

In [ ]:
# Ultralytics saves confusion matrix to runs/detect/val/
conf_matrix_img = BASE_DIR / 'runs' / 'detector' / 'confusion_matrix.png'
if not conf_matrix_img.exists():
    # Try val subfolder
    candidates = list((BASE_DIR / 'runs' / 'detector').rglob('confusion_matrix.png'))
    conf_matrix_img = candidates[0] if candidates else None

if conf_matrix_img:
    img = cv2.cvtColor(cv2.imread(str(conf_matrix_img)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix — Stage 1 Detector')
    plt.show()
else:
    print('Confusion matrix image not found in runs/detector/')